<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 🗺️ EarthDaily Agriculture - Medium Resolution Time Series Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
from earthdaily.agriculture.extractors.VTS_functions import MRTSExtractor
import pandas as pd
import os
from datetime import datetime
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 - Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from flm_extractor.py**

### 🗺️ Configuration

In [ ]:
mrts_extractor = MRTSExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
    workflow_ref=manager  # For token refresh
)

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id", "start_date": "sowingDate"}

# Parameters configuration
mrts_extractor.setup_mrts_parameters(
    start_date='2025-11-01',
    end_date='2026-02-28',
    sensors=None,
    vegetation_index='S2REP',
    aggregation='average',
    smoothing_method='Whittaker',
    apply_denoiser=True,
    apply_end_of_curve=True,
    clear_cover_min=95,
    output_saturation=True,
    extract_raw_datasets=True,
    compute_temporal_consistency=True,
    temporal_consistency_threshold={"Ndvi": 0.06, "Lai": 0.3, "S2Rep": 2.5},
    mode='raw',
    historical_years=5,
    column_mapping=column_mapping
)

### Test api request

In [ ]:
# Use a real entity from the platform for testing LAI
# First, load entities filtered by crop to ensure 'crop.id' is available
manager.load_seasonfields(
    sowing_date_gte="2025-07-01",
    crop_id="WINTER_OSR",
)

# Pick a test entity from the loaded list
seasonfield_data = manager.sfd_list.iloc[6]
print(f"Test entity: {seasonfield_data['id']}")
print(f"Crop: {seasonfield_data['crop.id']}")

Get MRTS based on geometry

In [ ]:
print("\n--- get MR vegetation time series ---")
try:
    result = mrts_extractor.get_mrts_api(
        entity_data=seasonfield_data
    )
    
    print("✅ Vegetation time series data retrieved successfully!")
    if isinstance(result, dict) and 'value' in result:
        records = result['value']
        print(f"Number of records: {len(records)}")
        if records:
            print(f"\n📊 First record:")
            for key, value in list(records[0].items())[:5]:
                print(f"  {key}: {value}")
    else:
        print(f"Response: {result}")
    
except Exception as e:
    print(f"❌ API call failed: {e}")
    print(f"📄 API Error Response: {e.response.text}")  
    import traceback
    traceback.print_exc()

Test mrts safe 

In [ ]:
print("\n--- Test: get_vegation_ts_safe ---")
safe_result = mrts_extractor.get_mrts_api_safe(seasonfield_data)
print(safe_result)

Format mrts json

In [ ]:
output = mrts_extractor.format_mrts_json(result, seasonfield_data)
print(output.columns)


### 🗺️ process_mrts_single_seasonfield

In [ ]:
# Use a real entity from the platform for single entity test
row = manager.sfd_list.iloc[4].to_dict()

print("\n--- get MR vegetation time series ---")
results = mrts_extractor.process_single_entity_mrts(
    row=row
)

# Check results
if results['error']:
    print(f"❌ Error: {results['error']['message']}")
else:
    print("✅ Data retrieved successfully")
    print(f"📊 Total records: {len(results['data'])}")
    display(results['data'].head())

In [ ]:
print(result)

### KPI Accumulation on single entity

Reconfigure MRTS with `kpi_filter` to compute an accumulation KPI over the extraction period.  
Returns a single row per entity with current value, historical average, and percent change.

In [ ]:
# Reconfigure with KPI accumulation
mrts_extractor.setup_mrts_parameters(
    start_date="2025-01-15",
    end_date="2026-08-31",
    vegetation_index="NDVI",
    mode="full",
    clear_cover_min=80,
    historical_years=5,
    column_mapping=column_mapping,
    kpi_filter={
         "kpi_name": "NDVI Accumulation",
         "aggregation": "top_accumulation",
         'value_column': 'smoothed_value',
         "threshold": 30
     }
)

# Run on the same single entity
kpi_result = mrts_extractor.process_single_entity_mrts(row=manager.sfd_list.iloc[4])

df = kpi_result["data"]
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
# print(df[['date', 'raw_value', 'smoothed_value']].to_string())

if kpi_result["error"]:
    print(f"Error: {kpi_result['error']['message']}")
else:
    print("KPI result (single row per entity):")
    display(kpi_result["data"])

### KPI with per-entity `years` (string and list)

Tests that `years` works both as a native list and as a comma-separated string
(the format produced by `normalize_with_metadata` when a list flows through a pipeline).  
Also validates that `column_mapping` can remap `years` to another source column (e.g. `historical_seasons`).

In [ ]:
import copy

# Setup MRTS with KPI + historical comparison
mrts_extractor.setup_mrts_parameters(
    start_date='2025-11-01',
    end_date='2026-02-28',
    vegetation_index='NDVI',
    mode='raw',
    historical_years=5,
    column_mapping=column_mapping,
    kpi_filter={
        'kpi_name': 'NDVI Accumulation',
        'aggregation': 'accumulation',
    }
)

# --- Test 1: years as a list (native format) ---
row_list = copy.deepcopy(row)
row_list['years'] = [2024, 2023, 2022]

result_list = mrts_extractor.process_single_entity_mrts(row=row_list)
if result_list['error']:
    print(f'Test 1 (list) - Error: {result_list["error"]["message"]}')
else:
    df = result_list['data']
    print('Test 1 (years as list [2024, 2023, 2022]):')
    print(f'  historical_num_years: {df["historical_num_years"].iloc[0]}')
    print(f'  current_value: {df["current_value"].iloc[0]}, historical_avg: {df["historical_avg"].iloc[0]}')

# --- Test 2: years as comma-separated string (pipeline format) ---
row_str = copy.deepcopy(row)
row_str['years'] = '2024,2023,2022'

result_str = mrts_extractor.process_single_entity_mrts(row=row_str)
if result_str['error']:
    print(f'Test 2 (string) - Error: {result_str["error"]["message"]}')
else:
    df = result_str['data']
    print()
    print('Test 2 (years as string "2024,2023,2022"):')
    print(f'  historical_num_years: {df["historical_num_years"].iloc[0]}')
    print(f'  current_value: {df["current_value"].iloc[0]}, historical_avg: {df["historical_avg"].iloc[0]}')

# --- Test 3: years via column_mapping (remap historical_seasons -> years) ---
mrts_extractor.setup_mrts_parameters(
    start_date='2025-11-01',
    end_date='2026-02-28',
    vegetation_index='NDVI',
    mode='raw',
    historical_years=5,
    column_mapping={**column_mapping, 'years': 'historical_seasons'},
    kpi_filter={
        'kpi_name': 'NDVI Accumulation',
        'aggregation': 'accumulation',
    }
)

row_mapped = copy.deepcopy(row)
row_mapped['historical_seasons'] = '2024,2023,2022'  # simulates CropID pipeline output

result_mapped = mrts_extractor.process_single_entity_mrts(row=row_mapped)
if result_mapped['error']:
    print(f'Test 3 (column_mapping) - Error: {result_mapped["error"]["message"]}')
else:
    df = result_mapped['data']
    print()
    print('Test 3 (years mapped from historical_seasons column):')
    print(f'  historical_num_years: {df["historical_num_years"].iloc[0]}')
    print(f'  current_value: {df["current_value"].iloc[0]}, historical_avg: {df["historical_avg"].iloc[0]}')


### Debug: verify API date range for KPI historical comparison

Calls the API directly to check the actual date range returned,
confirming historical data was fetched for KPI comparison.

In [ ]:
import copy
from datetime import datetime

# Reconfigure with KPI + historical years
mrts_extractor.setup_mrts_parameters(
    start_date='2025-09-01',
    end_date='2025-12-30',
    vegetation_index='NDVI',
    mode='raw',
    historical_years=5,
    column_mapping=column_mapping,
    kpi_filter={
        'kpi_name': 'NDVI Accumulation',
        'aggregation': 'accumulation',
    }
)

params = mrts_extractor.mrts_params
start_date = mrts_extractor.get_entity_value(row, "start_date") or params.get("start_date")
end_date = mrts_extractor.get_entity_value(row, "end_date") or params.get("end_date")
historical_years = params.get("historical_years", 0)

print("Date resolution:")
print(f"  start_date column:   {mrts_extractor.get_mapped_column('start_date')}")
print(f"  end_date column:     {mrts_extractor.get_mapped_column('end_date')}")
print(f"  resolved start_date: {start_date}")
print(f"  resolved end_date:   {end_date}")
print(f"  historical_years:    {historical_years}")

# Replicate the date expansion from process_single_entity_mrts
start_dt = datetime.strptime(start_date, "%Y-%m-%d")
if isinstance(historical_years, list):
    effective_lookback = start_dt.year - min(historical_years)
else:
    effective_lookback = historical_years
expanded_start = start_dt.replace(year=start_dt.year - effective_lookback).strftime("%Y-%m-%d")
print(f"  expanded start_date: {expanded_start} ({effective_lookback} years lookback)")

# Inject expanded dates into row (same as process_single_entity_mrts does)
row_debug = copy.deepcopy(row)
row_debug['years'] = [2024, 2023, 2022]
start_col = mrts_extractor.get_mapped_column("start_date")
end_col = mrts_extractor.get_mapped_column("end_date")
row_debug[start_col] = expanded_start
row_debug[end_col] = end_date
print(f"  row[{start_col}] = {row_debug[start_col]}")
print(f"  row[{end_col}] = {row_debug[end_col]}")

# Call API with expanded dates
mrts_extractor.ensure_token_valid()
safe = mrts_extractor.get_mrts_api_safe(row_debug)
print()
print(f"API success: {safe['success']}")
if safe["data"]:
    raw_data = safe["data"].get("rawData") or []
    smoothed_data = safe["data"].get("smoothedData") or []
    print(f"  rawData records:      {len(raw_data)}")
    print(f"  smoothedData records: {len(smoothed_data)}")
    if raw_data:
        dates = sorted([r["date"] for r in raw_data])
        print(f"  date range:           {dates[0]} to {dates[-1]}")
        from collections import Counter
        year_counts = Counter(d[:4] for d in dates)
        for yr in sorted(year_counts):
            print(f"    {yr}: {year_counts[yr]} records")
else:
    print(f"  error: {safe.get('error')}")


### Manual KPI verification: export time series then compute KPI

Extracts the full raw time series (with historical years) as CSV for manual inspection,
then computes the KPI on the same data so results can be cross-checked in a spreadsheet.

In [ ]:
import os
from earthdaily.agriculture.core.api_utils import filter_timeseries_kpi, format_kpi_results

# KPI period: the window you want to compare across years
kpi_start = "2025-06-01"
kpi_end   = "2025-06-30"

# Historical lookback: how many past years to include.
# The API start_date must be pushed back manually because date expansion
# only triggers when kpi_filter is set. For a CSV export (kpi_filter=None),
# compute the expanded start yourself:
#   api_start = kpi_start with year minus N  (e.g. 2025 - 5 = 2020)
historical_n = 5
api_start = f"{int(kpi_start[:4]) - historical_n}{kpi_start[4:]}"
print(f"KPI period:  {kpi_start} to {kpi_end}")
print(f"API request: {api_start} to {kpi_end} ({historical_n} years lookback)")

# Column mapping to force extraction on defined dates (not entity sowing date)
column_mapping_fixed = {"crop": "crop.id", "start_date": "start"}

# Step 1: Extract full time series WITHOUT KPI (get all rows)
mrts_extractor.setup_mrts_parameters(
    start_date=api_start,
    end_date=kpi_end,
    vegetation_index="NDVI",
    mode="full",
    historical_years=0,  # already expanded manually
    column_mapping=column_mapping_fixed,
    kpi_filter=None,
)

ts_result = mrts_extractor.process_single_entity_mrts(row=row)
if ts_result["error"]:
    print(f"Error: {ts_result['error']['message']}")
else:
    ts_df = ts_result["data"]
    print(f"Time series shape: {ts_df.shape}")
    print(f"Columns: {ts_df.columns.tolist()}")
    print(f"Date range: {ts_df['date'].min()} to {ts_df['date'].max()}")
    print()

    # Records per year
    from collections import Counter
    year_counts = Counter(ts_df["date"].dt.year)
    for yr in sorted(year_counts):
        print(f"  {yr}: {year_counts[yr]} records")

    # Export to CSV for manual verification
    csv_path = os.path.join(manager.output_result_dir, "kpi_manual_check_timeseries.csv")
    ts_df.to_csv(csv_path, index=False)
    print()
    print(f"Exported to: {csv_path}")
    display(ts_df.head(10))

    # Step 2: Compute KPI on the exported data using the CURRENT period dates
    # filter_timeseries_kpi slices current vs historical from the full dataset
    print()
    print("=" * 60)
    print(f"KPI computation (current period: {kpi_start} to {kpi_end})")
    print("=" * 60)

    kpi_configs = [
        ("Accumulation",        "accumulation",     None),
        ("Top-30 Accumulation", "top_accumulation", 30),
        ("Average",             "average",           None),
        ("Max",                 "max",               None),
        ("Count > 0.5",         "count_gt",          0.5),
    ]
# KPI compute over all years
    for name, agg, thresh in kpi_configs:
        kpi = filter_timeseries_kpi(
            timeseries_df=ts_df,
            start_date=kpi_start,
            end_date=kpi_end,
            kpi_name=name,
            aggregation=agg,
            threshold=thresh,
            years="ALL",
            date_column="date",
            value_column="smoothed_value",
        )
        curr = kpi["current_period"]["value"]
        hist = kpi["historical_avg"]["value"]
        nyrs = kpi["historical_avg"]["num_years"]
        pct = kpi["comparison"]["percent_change"]
        print(f"  {name:25s}  current={curr}  hist_avg={hist} ({nyrs} yrs)  change={pct}%")

# KPI compute 2 selected years
    for name, agg, thresh in kpi_configs:
        kpi = filter_timeseries_kpi(
            timeseries_df=ts_df,
            start_date=kpi_start,
            end_date=kpi_end,
            kpi_name=name,
            aggregation=agg,
            threshold=thresh,
            years=[2023,2024],
            date_column="date",
            value_column="smoothed_value",
        )
        curr = kpi["current_period"]["value"]
        hist = kpi["historical_avg"]["value"]
        nyrs = kpi["historical_avg"]["num_years"]
        pct = kpi["comparison"]["percent_change"]
        print("Kpi over specific years")
        print(f"  {name:25s}  current={curr}  hist_avg={hist} ({nyrs} yrs)  change={pct}%")


### 🗺️ process_mrts_bulk_extraction_parallel

In [ ]:
# Subset test
test_subset = manager.sfd_list.head(20).copy()  # The number head(X) means the first X fields 


# Bulk launch
results = mrts_extractor.process_mrts_bulk_parallel(
    entity_list=test_subset,
    output_path=manager.output_result_dir,
    max_workers=20
)

# 🔍 DEBUG: Check what keys are actually in results
print("\n🔍 Keys in results:", list(results.keys()))
print("\n🔍 Results structure:")
for key, value in results.items():
    print(f"  {key}: {type(value)}")

# Try to access errors with the correct key
possible_error_keys = ['global_errors', 'errors', 'error_list', 'failed']
for key in possible_error_keys:
    if key in results:
        print(f"\n✅ Found errors under key '{key}':")
        errors = results[key]
        if errors:
            print(f"❌ Total errors: {len(errors)}")
            for i, error in enumerate(errors[:3]):  # Show first 3
                print(f"\nError {i+1}:")
                print(f"  {error}")
        break